# Prithvi EO 2.0 — Model Overview

[Prithvi EO 2.0](https://huggingface.co/ibm-nasa-geospatial) is a family of geospatial foundation models jointly developed by IBM Research and NASA. Trained on 4.2M multi-spectral, multi-temporal image chips from NASA's Harmonized Landsat and Sentinel-2 (HLS) dataset, Prithvi provides strong spatial-spectral representations that transfer well to downstream tasks with small labeled datasets.

**Model variants:**
- `Prithvi-EO-2.0-300M` — 300M parameters, fast fine-tuning
- `Prithvi-EO-2.0-600M` — 600M parameters, best accuracy on GEO-Bench

**Architecture:** Vision Transformer (ViT) with temporal attention — processes sequences of multi-spectral HLS patches (6 bands: Blue, Green, Red, Narrow-NIR, SWIR-1, SWIR-2) at multiple timestamps simultaneously.

**Why use Prithvi?** When you have a small labeled EO dataset and need strong transfer learning. Fine-tuning Prithvi with LoRA requires only a fraction of the labeled data needed to train from scratch.

## References
- HuggingFace: https://huggingface.co/ibm-nasa-geospatial
- Paper: https://arxiv.org/abs/2310.18660
- GEO-Bench results: https://github.com/ServiceNow/geo-bench

## 0. Load HuggingFace Token

In [ ]:
import sys
import os

# Load HF_TOKEN from .env file (see .env.example in repo root)
sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
    if HF_TOKEN:
        print("HF_TOKEN loaded from .env")
    else:
        print("HF_TOKEN not set. Set it in .env if model access is restricted.")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print(f"HF_TOKEN from environment: {'set' if HF_TOKEN else 'not set'}")

## 1. Load the Model from HuggingFace

In [ ]:
from transformers import AutoConfig, AutoModel
import torch

model_id = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"

config = AutoConfig.from_pretrained(
    model_id,
    token=HF_TOKEN or None,
    trust_remote_code=True,
)
print("Model config loaded.")
print(f"  Model type: {config.model_type}")
print(f"  Image size: {getattr(config, 'img_size', 'N/A')}")
print(f"  Patch size: {getattr(config, 'patch_size', 'N/A')}")
print(f"  Num bands:  {getattr(config, 'num_frames', 'N/A')} frames x {getattr(config, 'in_chans', 'N/A')} bands")

In [ ]:
# Load the model (downloads ~1.2 GB for 300M on first run)
model = AutoModel.from_pretrained(
    model_id,
    token=HF_TOKEN or None,
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Understand the Input Format

Prithvi expects multi-temporal HLS imagery:
- Shape: `[batch, time, channels, height, width]`
- `time`: number of timestamps (typically 1–3)
- `channels`: 6 (Blue, Green, Red, Narrow-NIR, SWIR-1, SWIR-2)
- `height x width`: 224×224 px (native patch size)
- Values: HLS surface reflectance scaled to [0, 10000]

In [ ]:
import torch

batch_size = 2
num_frames = 3   # 3 timestamps
num_channels = 6 # HLS bands
img_size = 224

# Simulate a batch of multi-temporal HLS patches (normalized reflectance)
dummy_input = torch.randn(batch_size, num_frames, num_channels, img_size, img_size)
print(f"Input shape: {dummy_input.shape}")
print(f"  [{batch_size} samples, {num_frames} timestamps, {num_channels} bands, {img_size}x{img_size} px]")

## 3. Forward Pass — Extract Patch Embeddings

In [ ]:
with torch.no_grad():
    outputs = model(pixel_values=dummy_input)

print("Output keys:", list(outputs.keys()) if hasattr(outputs, 'keys') else type(outputs))

if hasattr(outputs, 'last_hidden_state'):
    embeddings = outputs.last_hidden_state
    print(f"Embedding shape: {embeddings.shape}")
    print(f"  [{batch_size} samples, N_patches, embedding_dim]")

## 4. Model Architecture Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Prithvi EO 2.0-300M Architecture Summary")
print("=" * 45)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Model size (FP32):    {total_params * 4 / 1e9:.2f} GB")
print(f"Model size (FP16):    {total_params * 2 / 1e9:.2f} GB")

## 5. Available Prithvi Variants on HuggingFace

| Model | Parameters | Best for |
|---|---|---|
| `Prithvi-EO-2.0-300M` | 307M | Faster fine-tuning, less VRAM |
| `Prithvi-EO-2.0-600M` | 636M | Highest accuracy on GEO-Bench |
| `Prithvi-EO-1.0-100M` | 100M | Lightest, for edge/CPU use |

All variants are at: https://huggingface.co/ibm-nasa-geospatial

Continue to `1-feature_extraction.ipynb` for PCA/UMAP visualization of embeddings.